# 06 · Season Projections

**Goal:** Project every team's final record and playoff odds from this week's Elo ratings, and measure how accurate those projections have been from the same week in past seasons.

**Data:** `nflmodel.data.read_games()`, Elo from `nflmodel.elo.run_elo` with the tuned `DEFAULT_PARAMS`

**Contents:** 0. Setup & parameters · 1. Load & Elo · 2. Projection engine · 3. 2026 projection ·
4. Backtest · 5. League table

Each team keeps the wins it already has. Its remaining games are simulated `N_SIMS` times, with each game's win probability coming from the two teams' current Elo ratings. Ratings stay fixed for the rest of the season. Playoff odds use 4 division winners plus 3 wild cards per conference. Ties in the standings are broken at random rather than with the NFL tiebreakers. The backtest repeats the same projection from the same week in each of the previous 5 seasons and compares it to how those seasons ended.

The projection engine (`project_season`, `team_wins`) lives in `src/nflmodel/projections.py`. The Carolina-specific outlook that used to be Sections 6–7 is now in `analytical_summaries/01_wp_calibration_gm/carolina_outlook_gm.ipynb`.

In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
from pathlib import Path

from nflmodel.data import read_games
from nflmodel.elo import run_elo, DEFAULT_PARAMS, team_history
from nflmodel.projections import project_season, team_wins

In [2]:
# =============================
# Section 0: setup & parameters
# =============================

SEASON = 2026
THROUGH_WEEK = 7                     # last completed week; projections start the week after
BACKTEST_SEASONS = range(2021, 2026)  # 5 prior seasons, all 17-game schedules
N_SIMS = 10_000
FORECAST_DIR = Path("../forecasts")


In [3]:
# =============================
# Section 1: load & Elo
# =============================

games = read_games()
elo = run_elo(games, DEFAULT_PARAMS)
history = team_history(elo)

latest = games[(games["season"] == SEASON) & games["home_win"].notna()]["week"].max()
print(f"Latest week with scores: {latest}  |  projecting from after Week {THROUGH_WEEK}")

Latest week with scores: 7  |  projecting from after Week 7


## Section 2: projection engine

`project_season` and `team_wins` are imported from `nflmodel.projections` (moved there unchanged so the analytical summaries can reuse them). `project_season` takes the `games` table and Elo `history` as its first two arguments.

In [4]:
# =============================
# Section 3: 2026 projection
# =============================

proj, sims, sim_playoff = project_season(games, history, SEASON, THROUGH_WEEK, DEFAULT_PARAMS, N_SIMS)
proj = proj.sort_values("proj_wins", ascending=False)

# Sanity checks: every simulated season has 14 playoff teams and 8 division winners
print(f"Playoff spots per season: {proj['playoff_pct'].sum():.1f}  |  division titles: {proj['division_pct'].sum():.1f}")

proj.to_csv(FORECAST_DIR / f"{SEASON}_thru_wk{THROUGH_WEEK:02d}_projections.csv")
proj.round(2)

Playoff spots per season: 14.0  |  division titles: 8.0


,division,elo,wins,losses,proj_wins,low,high,division_pct,playoff_pct
team,,,,,,,,,
MIN,NFC North,1631.29,6.0,0.0,13.12,11.0,15.0,0.79,0.99
SF,NFC West,1618.04,6.0,1.0,12.53,11.0,14.0,0.72,0.98
BUF,AFC East,1639.46,5.0,1.0,12.38,10.0,14.0,0.83,0.96
DEN,AFC West,1616.09,5.0,2.0,11.80,10.0,14.0,0.63,0.93
JAX,AFC South,1602.62,4.0,2.0,11.49,10.0,13.0,0.95,0.96
CIN,AFC North,1561.68,4.0,2.0,11.11,9.0,13.0,0.45,0.83
SEA,NFC West,1658.10,4.0,3.0,11.09,9.0,13.0,0.25,0.87
KC,AFC West,1558.01,5.0,1.0,11.09,9.0,13.0,0.37,0.84
PIT,AFC North,1554.27,5.0,2.0,10.76,9.0,13.0,0.34,0.78


In [5]:
# =============================
# Section 4: backtest
# =============================

def rmse(errors):
    """Root mean squared error: the typical miss, in wins."""
    return np.sqrt((errors ** 2).mean())

# 1. Project each past season from the same week, then attach how it actually ended
rows = []
for s in BACKTEST_SEASONS:
    p, _, _ = project_season(games, history, s, THROUGH_WEEK, DEFAULT_PARAMS, N_SIMS)
    season_games = games[games["season"] == s]
    final = team_wins(season_games[season_games["game_type"] == "REG"])
    postseason = season_games[season_games["is_playoff"]]

    p["actual_wins"] = final["sum"].values
    p["pace_wins"] = p["wins"] / (p["wins"] + p["losses"]) * final["count"].values  # naive baseline
    p["made_playoffs"] = p.index.isin(set(postseason["home_team"]) | set(postseason["away_team"]))
    p["season"] = s
    rows.append(p.reset_index())

backtest = pd.concat(rows, ignore_index=True)
backtest["error"] = backtest["proj_wins"] - backtest["actual_wins"]
backtest["in_range"] = backtest["actual_wins"].between(backtest["low"], backtest["high"])

# 2. Win-total accuracy by season, plus all seasons together
def accuracy(g):
    return pd.Series({
        "teams": len(g),
        "rmse_elo": rmse(g["error"]),
        "rmse_pace": rmse(g["pace_wins"] - g["actual_wins"]),
        "within_1": (g["error"].abs() <= 1).mean(),
        "within_2": (g["error"].abs() <= 2).mean(),
        "in_80_range": g["in_range"].mean(),
    })

by_season = backtest.groupby("season").apply(accuracy)
by_season.loc["All"] = accuracy(backtest)
RMSE = by_season.loc["All", "rmse_elo"]

# 3. Playoff odds calibration: of teams we gave X% odds, how many actually made it?
backtest["odds_bin"] = pd.cut(backtest["playoff_pct"], [-0.01, 0.1, 0.5, 0.9, 1.0],
                              labels=["Under 10%", "10-50%", "50-90%", "Over 90%"])
calibration = (backtest.groupby("odds_bin", observed=True)
               .agg(teams=("team", "size"),
                    avg_projected=("playoff_pct", "mean"),
                    actually_made=("made_playoffs", "mean")))

print(f"Backtest RMSE: {RMSE:.2f} wins (Elo) vs {by_season.loc['All', 'rmse_pace']:.2f} (current pace)")
calibration.round(2)

Backtest RMSE: 1.96 wins (Elo) vs 2.51 (current pace)


,teams,avg_projected,actually_made
odds_bin,,,
Under 10%,47,0.03,0.06
10-50%,48,0.29,0.33
50-90%,38,0.75,0.74
Over 90%,27,0.96,0.85


## Section 5: League table

Projected final record for every team, grouped by division. **Range** is the middle 80% of simulated outcomes. **Typical miss** is the backtest RMSE: in past seasons, projections from this week missed the final win total by about this much.

In [6]:
# =============================
# Section 5: league table
# =============================

def display_table(p):
    """Projection rows formatted for a reader: record, projection, range, odds."""
    t = pd.DataFrame({
        "Division": p["division"],
        "Elo": p["elo"].round(0).astype(int),
        "Record": p["wins"].map("{:g}".format) + "-" + p["losses"].map("{:g}".format),
        "Proj. wins": p["proj_wins"],
        "80% range": p["low"].map("{:g}".format) + "-" + p["high"].map("{:g}".format),
        "Division %": p["division_pct"],
        "Playoff %": p["playoff_pct"],
    })
    return (t.style
             .format({"Proj. wins": "{:.1f}", "Division %": "{:.0%}", "Playoff %": "{:.0%}"})
             .background_gradient(subset=["Playoff %", "Division %"], cmap="RdYlGn", vmin=0, vmax=1)
             .background_gradient(subset=["Proj. wins"], cmap="RdYlGn", vmin=3, vmax=14))

league = proj.sort_values(["division", "proj_wins"], ascending=[True, False])
display_table(league).set_caption(
    f"{SEASON} projected records after Week {THROUGH_WEEK} · typical miss ±{RMSE:.1f} wins")

,Division,Elo,Record,Proj. wins,80% range,Division %,Playoff %
team,,,,,,,
BUF,AFC East,1639,5-1,12.4,10-14,83%,96%
NE,AFC East,1591,4-3,10.2,8-12,17%,67%
NYJ,AFC East,1369,2-5,5.0,3-7,0%,0%
MIA,AFC East,1385,1-5,4.4,3-6,0%,0%
CIN,AFC North,1562,4-2,11.1,9-13,45%,83%
PIT,AFC North,1554,5-2,10.8,9-13,34%,78%
CLE,AFC North,1473,5-2,9.8,8-12,16%,55%
BAL,AFC North,1508,4-3,8.8,7-11,6%,31%
JAX,AFC South,1603,4-2,11.5,10-13,95%,96%


In [7]:
# Backtest summary: how accurate projections from this week have been
(by_season.style
    .format({"teams": "{:.0f}", "rmse_elo": "{:.2f}", "rmse_pace": "{:.2f}",
             "within_1": "{:.0%}", "within_2": "{:.0%}", "in_80_range": "{:.0%}"})
    .set_caption(f"Projections made after Week {THROUGH_WEEK} vs final records "
                 f"(pace = current win % carried forward)"))

,teams,rmse_elo,rmse_pace,within_1,within_2,in_80_range
season,,,,,,
2021,32,1.67,2.74,47%,78%,81%
2022,32,2.30,2.93,38%,66%,66%
2023,32,1.54,1.92,34%,78%,88%
2024,32,1.85,1.95,41%,69%,81%
2025,32,2.30,2.79,28%,50%,66%
All,160,1.96,2.51,38%,68%,76%
